# Unit 11 (V2) · Optional lab: the same checks with a real AI model

<a href="https://colab.research.google.com/github/dinbav/digital-experiments-course/blob/main/notebooks/v2-ai-eval-real-model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The simulation notebook showed what *can* go wrong when the treatment is an AI
system. Here you watch it happen with real, open-source language models
running inside Colab: the same question answered differently every time, two
AI judges disagreeing, a judge that likes long answers, and a "vendor update"
that moves the scores.

**After this notebook you can:** run a small prompt A/B test on a real model, see
non-determinism with your own eyes, measure agreement between two AI judges,
check a judge for length bias, and compare cost between arms.

**Optional.** Everything here is also covered, with guaranteed numbers, in the
[simulation demo](https://colab.research.google.com/github/dinbav/digital-experiments-course/blob/main/solutions/v2-ai-eval-demo.ipynb). Do this lab if you are curious or your
project's treatment is an AI system.

## Before you start

1. **Use a free GPU:** in Colab, *Runtime -> Change runtime type -> T4 GPU -> Save*.
   On a GPU the whole notebook takes about **5 minutes**. On a CPU it still works,
   but automatically uses fewer questions and takes about 15-20 minutes.
2. **No account and no API key.** The models are downloaded from
   [Hugging Face](https://huggingface.co), a public library of open-source models:
   - `Qwen/Qwen2.5-0.5B-Instruct` - answers the customer questions (about 1 GB)
   - `Qwen/Qwen2.5-1.5B-Instruct` - acts as the AI judge, and later as the "updated model" (about 3 GB)

   Both are released under the Apache 2.0 open-source licence. The first download
   takes a few minutes; you may see progress bars and harmless warnings.
3. **Your numbers will differ** from a classmate's, and from run to run. That is
   not a bug - it is lesson one. This notebook has no `assert` checks for the same
   reason: you read the results and judge them yourself.

## 1. The question

**A support team wants to test two system prompts for its AI assistant.**

- **Control:** "answer in one or two short sentences"
- **Treatment:** "give a thorough, step-by-step answer"

Which one is better? And can an AI judge be trusted to say so?

## 2. Setup

The first cell loads the libraries and checks for a GPU. Colab already has
`transformers` and `torch`; if an import fails, remove the `#` in front of
`%pip` and run the cell again.

In [ ]:
# %pip install -q transformers accelerate

import re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import pipeline, set_seed

RANDOM_SEED = 42
set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = 0 if torch.cuda.is_available() else -1
QUICK = DEVICE == -1          # on a CPU, use fewer questions so it finishes in reasonable time
print('Running on', 'GPU - good.' if DEVICE == 0 else
      'CPU - this will be slow. Runtime -> Change runtime type -> T4 GPU is recommended.')

GENERATOR = 'Qwen/Qwen2.5-0.5B-Instruct'
JUDGE = 'Qwen/Qwen2.5-1.5B-Instruct'

Now download and load the two models. This is the slow cell (a few minutes the
first time).

In [ ]:
gen = pipeline('text-generation', model=GENERATOR, device=DEVICE)
judge = pipeline('text-generation', model=JUDGE, device=DEVICE)
for p in (gen, judge):
    p.tokenizer.padding_side = 'left'   # needed to answer several questions in one batch

def ask(pipe, conversations, sample=False, max_new_tokens=160):
    '''Send a list of chat conversations to a model and return the answers as strings.
    sample=True: the model picks words at random in proportion to their probability (like a live product).
    sample=False: the model always picks the most likely word (repeatable).'''
    settings = dict(max_new_tokens=max_new_tokens, batch_size=8, return_full_text=False)
    settings.update(dict(do_sample=True, temperature=0.8, top_p=0.95) if sample else dict(do_sample=False))
    answers = []
    for out in pipe(conversations, **settings):
        text = out[0]['generated_text'] if isinstance(out, list) else out['generated_text']
        answers.append((text[-1]['content'] if isinstance(text, list) else text).strip())
    return answers

def n_tokens(text):
    '''Length in tokens - the unit AI providers charge by.'''
    return len(gen.tokenizer(text)['input_ids'])

print('Models loaded.')

## 3. The data

Twelve typical questions to an online store's support team, and the two system
prompts (the two arms of the experiment).

In [ ]:
QUESTIONS = [
    'How do I reset my password?',
    'Can I change the delivery address after I placed an order?',
    'My package says delivered but I cannot find it. What should I do?',
    'How long does a refund take?',
    'Can I return an item I bought on sale?',
    'How do I cancel my subscription?',
    'Why was my card declined?',
    'Do you ship internationally?',
    'How can I track my order?',
    'The item arrived damaged. What now?',
    'Can I use two discount codes on one order?',
    'How do I delete my account?',
]
if QUICK:
    QUESTIONS = QUESTIONS[:6]

PROMPTS = {
    'control': 'You are a customer-support assistant for an online store. '
               'Answer in one or two short sentences.',
    'treatment': 'You are a customer-support assistant for an online store. '
                 'Give a thorough, detailed answer with step-by-step instructions.',
}

def conversation(arm, question):
    return [{'role': 'system', 'content': PROMPTS[arm]}, {'role': 'user', 'content': question}]

print(len(QUESTIONS), 'questions x 2 arms')

## 4. The naive move - and the first surprise

The naive plan: ask each question once per arm, score it, compare. But ask the
**same** question to the **same** arm five times, with sampling switched on as a
live product would:

In [ ]:
q = QUESTIONS[2]
five = ask(gen, [conversation('treatment', q)] * 5, sample=True)
print('Question:', q, '\n')
for i, a in enumerate(five, 1):
    print(f'--- answer {i} ({len(a.split())} words) ---')
    print(a[:300] + ('...' if len(a) > 300 else ''), '\n')

Same input, five different answers - different wording, different length,
sometimes different advice. The "treatment" is not one fixed thing. Now switch
sampling off (always pick the most likely word):

In [ ]:
fixed = ask(gen, [conversation('treatment', q)] * 2, sample=False)
print('Identical answers with sampling off:', fixed[0] == fixed[1])

Switching sampling off (or fixing temperature and seed) makes an experiment
repeatable - but real users usually get the sampled version. Either way, write
down exactly which settings the treatment is: model name and version, prompt,
temperature. That *is* the treatment.

## 5. What actually happens

### 5.1 Run the experiment

Each question, each arm, two sampled answers - like two users asking the same
thing. We also record the time taken and the answer length in tokens (the unit
providers charge by).

In [ ]:
rows = []
for arm in PROMPTS:
    start = time.perf_counter()
    convs = [conversation(arm, q) for q in QUESTIONS for _ in range(2)]
    answers = ask(gen, convs, sample=True)
    seconds = (time.perf_counter() - start) / len(answers)
    for c, a in zip(convs, answers):
        rows.append({'arm': arm, 'question': c[1]['content'], 'answer': a,
                     'words': len(a.split()), 'tokens': n_tokens(a), 'seconds': seconds})
answers_df = pd.DataFrame(rows)
print(answers_df.groupby('arm')[['words', 'tokens', 'seconds']].mean().round(1))

### 5.2 Two AI judges

Both judges are the larger model, with two different **rubrics** (grading
instructions). Judge A is asked for general helpfulness. Judge B is told to
grade only whether the answer solves the problem, and not to reward length.
The judges answer with a single digit from 1 to 5; sampling is off, so their
scores are repeatable.

In [ ]:
RUBRICS = {
    'judge_a': 'You grade customer-support answers. Rate how helpful the answer is, '
               'from 1 (useless) to 5 (excellent). Reply with a single digit only.',
    'judge_b': 'You grade customer-support answers. Judge only whether the answer is correct and '
               'solves the customer problem. A short answer that solves the problem deserves a 5. '
               'Do not reward length or extra detail. Reply with a single digit from 1 to 5 only.',
}

def score(rubric, question, answer):
    return [{'role': 'system', 'content': rubric},
            {'role': 'user', 'content': f'Customer question: {question}\n\nAnswer: {answer}'}]

def first_digit(text):
    m = re.search(r'[1-5]', text)
    return int(m.group()) if m else np.nan

for name, rubric in RUBRICS.items():
    replies = ask(judge, [score(rubric, r.question, r.answer) for r in answers_df.itertuples()],
                  max_new_tokens=4)
    answers_df[name] = [first_digit(t) for t in replies]

print('Unreadable judge replies:', int(answers_df[['judge_a', 'judge_b']].isna().sum().sum()))
print(answers_df.groupby('arm')[['judge_a', 'judge_b']].mean().round(2))

### 5.3 Do the judges agree?

**Cohen's kappa** measures how often two raters give the same score, beyond
what chance alone would produce: 1 = perfect agreement, 0 = no better than chance.

In [ ]:
def cohens_kappa(a, b):
    a, b = pd.Series(a).astype(str), pd.Series(b).astype(str)
    observed = (a.values == b.values).mean()
    expected = sum((a == c).mean() * (b == c).mean() for c in set(a) | set(b))
    return (observed - expected) / (1 - expected) if expected < 1 else float('nan')

scored = answers_df.dropna(subset=['judge_a', 'judge_b'])
print(f'Exact agreement: {(scored.judge_a == scored.judge_b).mean():.0%}')
print(f'Cohen kappa:     {cohens_kappa(scored.judge_a, scored.judge_b):.2f}')

### 5.4 Does a judge reward length?

If a judge's score rises with answer length, part of any "win" for the wordier
arm is length, not quality.

In [ ]:
fig, ax = plt.subplots()
for name in RUBRICS:
    jitter = np.random.uniform(-0.1, 0.1, len(scored))
    ax.scatter(scored['words'], scored[name] + jitter, alpha=0.6, label=name)
    print(f'{name}: correlation of score with length = {scored[name].corr(scored["words"]):+.2f}')
ax.set_xlabel('answer length (words)')
ax.set_ylabel('judge score (1-5)')
ax.set_title('Does the score climb with length?')
ax.legend()
plt.show()

effect = scored.groupby('arm')[['judge_a', 'judge_b']].mean()
print('\nTreatment minus control, by judge:')
print((effect.loc['treatment'] - effect.loc['control']).round(2).to_string())

Compare the two judges' verdicts on the same answers. If they disagree about
which arm wins, the experiment's result depends on the rubric - which means the
rubric is part of your OEC and must be fixed, written down and checked against
human ratings *before* the test.

### 5.5 The cost guardrail

Providers charge per token, so the token ratio is a good proxy for the cost
ratio. Suppose the guardrail allows at most **1.5x** the control's cost.

In [ ]:
tokens = answers_df.groupby('arm')['tokens'].mean()
seconds = answers_df.groupby('arm')['seconds'].mean()
cost_ratio = tokens['treatment'] / tokens['control']
print(f'Tokens per answer - control {tokens["control"]:.0f}, treatment {tokens["treatment"]:.0f}: '
      f'cost ratio {cost_ratio:.1f}x')
print(f'Time per answer   - control {seconds["control"]:.2f}s, treatment {seconds["treatment"]:.2f}s')
print('Passes the 1.5x cost guardrail:', cost_ratio <= 1.5)

### 5.6 A "vendor update"

Imagine the provider silently swaps the model behind the treatment for a new
version halfway through the test. We play the update by answering the same
questions with the larger model, same treatment prompt, and grade both
versions with the length-blind judge B.

In [ ]:
convs = [conversation('treatment', q) for q in QUESTIONS for _ in range(2)]
v2_answers = ask(judge, convs, sample=True)       # the larger model plays 'version 2'
v2_scores = [first_digit(t) for t in ask(
    judge, [score(RUBRICS['judge_b'], c[1]['content'], a) for c, a in zip(convs, v2_answers)], max_new_tokens=4)]

before = answers_df.loc[answers_df.arm == 'treatment', 'judge_b'].mean()
after = np.nanmean(v2_scores)
print(f'Treatment score (judge B) - model v1: {before:.2f}, model v2: {after:.2f}')
print(f'Average length - v1: {answers_df.loc[answers_df.arm == "treatment", "words"].mean():.0f} words, '
      f'v2: {np.mean([len(a.split()) for a in v2_answers]):.0f} words')

Same prompt, same questions, different model - different answers, different
scores, different cost. If this happened in week two of a four-week test, the
pooled result would average two different treatments. Pin the model version
when you can, log it with every request, and plot results over time.

## 6. What you do about it

- **Define the treatment completely:** model and version, prompt, temperature. Log them per request.
- **Treat the judge as a rater:** write the rubric, measure agreement (kappa), check length bias, and calibrate on a sample of human ratings.
- **Put cost and latency in the guardrails** before the test.
- **Watch for drift:** pin versions and plot effects over time.

**A caution about this lab:** these are small models chosen to fit a free GPU.
Production systems use much larger models and judges. The *methods* carry
over; the specific numbers do not.

## Your answer

Double-click and write one or two sentences each.

**1.** Did the two judges agree on which prompt wins? What does that tell you about using an AI judge as your OEC?

**2.** Was either judge's score related to answer length? How would you correct for it?

**3.** Would the treatment prompt pass your guardrails? Which number decided it?

---

**Takeaway:** With real models, non-determinism, judge disagreement, length
bias, cost and drift are not hypotheticals - you can see all five in five minutes.

**Next:** [Task 11](https://github.com/dinbav/digital-experiments-course/blob/main/tasks/task-11.md) · [Simulation demo](https://colab.research.google.com/github/dinbav/digital-experiments-course/blob/main/solutions/v2-ai-eval-demo.ipynb) · [Simulation exercise](https://colab.research.google.com/github/dinbav/digital-experiments-course/blob/main/notebooks/v2-ai-eval-exercise.ipynb)

**Models:** [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) ·
[Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) (Apache 2.0)

**Further reading:** Kohavi, Tang & Xu, *Trustworthy Online Controlled Experiments* (2020), ch. 3 and ch. 21 (trust and guardrails); Zheng et al., *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*, NeurIPS 2023 (judge biases, including verbosity).